# Day 4 — Learning Lab: Storage Formats, Partitioning & Performance Levers

**GlobalMart Data Engineering Bootcamp — Day 4, Session: Partitioning Strategy & Storage Formats & Performance Levers**

---
## Section 1 — Project Overview

### What you will learn in this lab

This is a hands-on **learning lab**, not a quick exercise — every experiment below follows the same pattern so you build real intuition, not just watch numbers change:

```
Concept  →  Code  →  Run  →  Observe  →  Why did this happen?  →  Production use case
```

By the end you will have, with your own hands, on your own real (~1GB) dataset:

1. Read a raw CSV and seen exactly why CSV is a bad *storage* format even though it's a fine *exchange* format
2. Actually converted the same data into JSON, Parquet, and Delta, and measured (not just read about) the size and speed differences
3. Partitioned a Delta table by Year/Month and proven partition pruning with `.explain()` and the Spark UI
4. Created — and then fixed — the small-files problem yourself
5. Measured what caching actually buys you, with real timings
6. Compared clustering (`ZORDER`) against partitioning on a real query
7. Used `UPDATE`, `DELETE`, `MERGE`, schema evolution, and time travel — the features that make Delta more than "Parquet with extra steps"
8. Applied everything yourself in a final unassisted challenge

### The dataset

`retail_sales_dataset.csv` — a synthetic but realistic e-commerce sales dataset (25 columns: order/customer IDs, dates, geography, product, pricing, shipping, and returns). Generated locally via `generate_retail_sales_dataset.py` (sitting next to this notebook): **5,000,000 rows, ~976 MB**, spanning orders from 2022–2026 across 6 countries (India, USA, UK, Germany, Canada, Australia) and 8 product categories. Large enough that file-size, partitioning, and caching effects are actually visible — small sample CSVs can't show you this.

### Setup — upload the dataset to a Databricks Volume

1. In Databricks, go to **Catalog Explorer** → your catalog → your schema → **Volumes** → create a Volume if you don't already have one (e.g. `day4_lab`).
2. Upload `retail_sales_dataset.csv` into that Volume.
3. Update the path below to match your catalog/schema/volume, then run it.

> **Instructions:** Run each cell with **Shift + Enter**, top to bottom. Every code cell is preceded by a **Concept** explanation and followed by **Observe / Why** — read both, not just the code.

In [0]:
# ─── Setup: point at your Volume, and a scratch catalog/schema for the tables we'll build ──
# Replace these with your own catalog/schema/volume names.
CATALOG      = "naya_catalog"       # e.g. "gbmart"
LAB_SCHEMA   = "day4_lab"           # a scratch schema just for this lab — safe to drop entirely afterward
VOLUME_NAME  = "day4_lab"

volume_path = f"/Volumes/{CATALOG}/{LAB_SCHEMA}/{VOLUME_NAME}"
csv_path    = f"{volume_path}/retail_sales_dataset.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{LAB_SCHEMA}")

print(f"Volume path : {volume_path}")
print(f"CSV path    : {csv_path}")
print(f"Lab schema  : {CATALOG}.{LAB_SCHEMA}")

Volume path : /Volumes/naya_catalog/day4_lab/day4_lab
CSV path    : /Volumes/naya_catalog/day4_lab/day4_lab/retail_sales_dataset.csv
Lab schema  : naya_catalog.day4_lab


In [0]:
dbutils.fs.cp(
    "/Volumes/naya_catalog/day4_lab/day4_lab/7bd7de9a-1f6d-407a-ad62-9232926190b4_retailsalesdataset.csv",
    "/Volumes/naya_catalog/day4_lab/day4_lab/retail_sales_dataset.csv"
)

True

---
## Section 2 — Read CSV

### Concept

CSV is a **row-based, uncompressed, schema-less text format** — every value is just text between commas, with no metadata describing types. When Spark reads a CSV with `inferSchema=true`, it doesn't know the column types up front, so it does something expensive: **it scans the entire file once just to guess types** (is this column an integer, a double, a date, or just text?), then scans it again to actually load the data. Two full passes over gigabytes of text, before you've done anything with the data yet.

This is exactly **why CSV is not an ideal storage format**, even though it's a perfectly fine *exchange* format (easy to open in Excel, easy for any system to produce) — the schema-inference tax and the lack of compression are both structural, not something you can configure away.

### Code

In [0]:
import time

# inferSchema=True triggers the two-pass read described above — timing it
# makes the cost visible instead of theoretical.
start = time.time()

sales_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(csv_path)
)

# .count() forces Spark to actually execute the read (Spark is lazy — without
# an action like count(), nothing above this line has actually run yet)
row_count = sales_df.count()
elapsed_infer = time.time() - start

print(f"Rows: {row_count:,}")
print(f"Time with inferSchema=true (two passes): {elapsed_infer:.2f}s")

Rows: 5,000,000
Time with inferSchema=true (two passes): 38.51s


In [0]:
# ─── Compare: reading WITHOUT schema inference (single pass, everything as string) ──
start = time.time()
sales_df_nostring = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")   # every column lands as StringType
    .csv(csv_path)
)
row_count_2 = sales_df_nostring.count()
elapsed_nostring = time.time() - start

print(f"Time with inferSchema=false (one pass, all strings): {elapsed_nostring:.2f}s")
print(f"\nSpeed difference: inferSchema=true took "
      f"{elapsed_infer / max(elapsed_nostring, 0.001):.1f}x longer")
sales_df_nostring.printSchema()

Time with inferSchema=false (one pass, all strings): 6.50s

Speed difference: inferSchema=true took 5.9x longer
root
 |-- OrderID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- ShipDate: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Profit: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- OrderPriority: string (nullable = true)
 |-- ShippingM

### Observe

- The `inferSchema=true` read took noticeably longer than the `inferSchema=false` read — that's the extra full pass over the file to guess types.
- With `inferSchema=false`, `printSchema()` shows **every single column as `string`** — even `Quantity`, `UnitPrice`, `Year` — which is useless for anything numeric (you couldn't `SUM(Sales)` correctly on a string column without an explicit cast first).

### Why did this happen?

CSV carries **zero metadata**. Spark has no way to know `UnitPrice` is a decimal without either (a) reading the whole file once to guess, or (b) you telling it the schema explicitly up front with `.schema(mySchema)` (the fastest option in production, but requires you to already know and maintain the schema by hand). Parquet and Delta, which you'll use starting in Section 4, store the schema as metadata **inside the file itself** — no guessing, no extra pass, ever.

### Azure Databricks verification

Open **Catalog Explorer → your catalog → your schema → Volumes → your volume**. Click on `retail_sales_dataset.csv` — the file browser shows you its raw size on disk (should read close to 976 MB, matching what the generator script reported). Keep that number in mind; you'll compare it against the JSON/Parquet/Delta sizes in Section 4.

### Production use case

CSV is exactly how data usually *arrives* — a partner sends you a CSV export, a legacy system dumps a nightly CSV, a business user uploads one manually. That's fine for **Bronze** (land it raw, exactly as it arrived). The moment you're doing anything repeatedly against that data — Silver transformations, Gold aggregations, BI queries — you convert it to Parquet or Delta first. CSV is a landing format, never a working format.

---
## Section 3 — Explore Dataset

### Concept

Before transforming any dataset, a Data Engineer always profiles it first: what columns exist, what do sample rows actually look like, how many rows total, and are there any early red flags (nulls, unexpected value ranges, skewed distributions)? Skipping this step is how broken pipelines get built on wrong assumptions.

### Code

In [0]:
# We'll use the properly-typed DataFrame (with inferSchema) from here on.
print("Schema:")
sales_df.printSchema()

print(f"\nTotal rows: {sales_df.count():,}")
print(f"Total columns: {len(sales_df.columns)}")

Schema:
root
 |-- OrderID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- ShipDate: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- OrderPriority: string (nullable = true)
 |-- ShippingMode: string (nullable = true)
 |-- CustomerSegment: string (nullable = true)
 |-- Returned: string (null

In [0]:
display(sales_df.limit(10))

OrderID,CustomerID,OrderDate,ShipDate,Year,Month,Day,Country,State,City,Region,Category,SubCategory,ProductName,Brand,Quantity,UnitPrice,Discount,Sales,Profit,PaymentMethod,OrderPriority,ShippingMode,CustomerSegment,Returned
ORD-00000001,CUST-353098,2026-08-07,2026-08-10,2026,8,7,India,Maharashtra,Pune,APAC,Furniture,Chairs,ErgoChair Deluxe,Herman Miller,8,1105.95,0.0,8847.6,1205.06,Debit Card,High,Economy,Consumer,No
ORD-00000002,CUST-091200,2026-03-20,2026-03-26,2026,3,20,India,Maharashtra,Pune,APAC,Books,Non-Fiction,Self Help Guide,Scholastic,7,7.49,0.0,52.43,-4.12,Net Banking,Critical,Economy,Consumer,No
ORD-00000003,CUST-354673,2026-08-18,2026-08-22,2026,8,18,Germany,Berlin,Berlin,Europe,Beauty,Skincare,Vitamin C Serum,Nivea,1,37.09,0.2,29.67,4.23,Credit Card,Critical,Economy,Corporate,Yes
ORD-00000004,CUST-225688,2025-07-27,2025-07-30,2025,7,27,USA,Illinois,Chicago,North America,Books,Non-Fiction,History Chronicles,Penguin,6,58.12,0.1,313.85,-16.65,Debit Card,Medium,Same Day,Home Office,Yes
ORD-00000005,CUST-171837,2026-02-20,2026-02-28,2026,2,20,USA,Illinois,Chicago,North America,Sports,Team Sports,Soccer Ball,Adidas,9,97.59,0.2,702.65,6.91,Net Banking,Low,Standard,Home Office,No
ORD-00000006,CUST-402000,2026-05-06,2026-05-10,2026,5,6,Germany,Berlin,Berlin,Europe,Electronics,Headphones,AudioClear Buds,Samsung,8,227.86,0.0,1822.88,255.71,Net Banking,Critical,Economy,Home Office,No
ORD-00000007,CUST-327211,2026-10-01,2026-10-03,2026,10,1,UK,England,Birmingham,Europe,Home,Bedding,Memory Foam Pillow,Prestige,3,116.48,0.0,349.44,97.35,Cash on Delivery,Low,Express,Consumer,No
ORD-00000008,CUST-486382,2023-02-03,2023-02-13,2023,2,3,USA,Texas,Houston,North America,Sports,Team Sports,Basketball,Adidas,9,312.08,0.2,2246.98,173.16,Net Banking,Low,Same Day,Home Office,No
ORD-00000009,CUST-077179,2026-09-28,2026-10-08,2026,9,28,UK,England,Birmingham,Europe,Furniture,Chairs,ClassicWood Chair,IKEA,3,976.13,0.2,2342.71,430.27,Net Banking,Critical,Standard,Small Business,No
ORD-00000010,CUST-297678,2022-04-30,2022-05-09,2022,4,30,Australia,Queensland,Brisbane,APAC,Beauty,Haircare,Hair Straightener,L'Oreal,6,42.44,0.0,254.64,-5.83,UPI,High,Standard,Small Business,No


In [0]:
# Quick distribution checks — the kind of thing you'd always do before trusting a new dataset
display(sales_df.groupBy("Country").count().orderBy("Country"))
display(sales_df.groupBy("Category").count().orderBy("Category"))
display(sales_df.groupBy("Year", "Month").count().orderBy("Year", "Month"))

Country,count
Australia,833693
Canada,833275
Germany,832771
India,834243
UK,832470
USA,833548


Category,count
Beauty,624803
Books,624840
Clothing,625310
Electronics,625230
Furniture,624628
Grocery,624311
Home,624852
Sports,626026


Year,Month,count
2022,1,85169
2022,2,76715
2022,3,85060
2022,4,82130
2022,5,84585
2022,6,82049
2022,7,85285
2022,8,84871
2022,9,82093
2022,10,84858


### Observe

- Row and column counts confirm the file loaded completely (5,000,000 rows, 25 columns) and matches what the generator script reported.
- `Country` and `Category` counts should look roughly even across their categories — that's the synthetic generator's `random.choice()` at work, not a real-world skew (a real retail dataset would usually be far more concentrated in a few countries/categories).
- The Year/Month breakdown confirms the full 2022–2026 date range actually landed in the data, which matters for Section 5's partitioning demo.

### Why did this happen?

This is exactly what exploratory profiling is *for* — confirming the data matches what you expect (row count, schema, date coverage) before you build anything on top of it. A single unexpected `NULL`-heavy column or a date range that's wrong by a year is far cheaper to catch here than after a downstream Gold table has been built on it.

### Production use case

Every real Bronze-to-Silver pipeline has an exploration/validation step like this — sometimes automated as data-quality checks (row count thresholds, null-percentage checks, expected value ranges), sometimes manual during initial pipeline design. It's the same instinct either way: know your data before you transform it.

---
## Section 4 — Storage Format Comparison

### Concept

We're not going to just *talk about* Parquet and Delta being better than CSV — we're going to **actually convert this real dataset into JSON, Parquet, and Delta**, and measure the differences ourselves.

- **JSON**: still row-based and text, but at least self-describing (each record carries its own field names) — still no compression, still no columnar layout.
- **Parquet**: a **columnar** binary format. Instead of storing row-by-row (`OrderID,CustomerID,...` then next row `OrderID,CustomerID,...`), it stores column-by-column (all `OrderID`s together, then all `CustomerID`s together, etc.). This matters enormously: if a query only needs 3 of your 25 columns, Parquet can skip reading the other 22 entirely. It also compresses far better, because values within one column (e.g. all `Country` values) are far more repetitive than values across a mixed row.
- **Delta**: **Parquet files, plus a transaction log** (`_delta_log/`) recording every write as a versioned, atomic commit. That log is what gives Delta ACID transactions, time travel, and reliable concurrent writes — Parquet alone has none of that; Delta is Parquet's storage format plus a reliability layer on top.

### Code — convert the same data into all three formats

In [0]:
json_path    = f"{volume_path}/formats/sales_json"
parquet_path = f"{volume_path}/formats/sales_parquet"

# JSON — still row-based, but self-describing
sales_df.write.mode("overwrite").json(json_path)

# Parquet — columnar, compressed
sales_df.write.mode("overwrite").parquet(parquet_path)

# Delta — Parquet files + transaction log, written as a managed table so we
# can query it by name and inspect it in Catalog Explorer
sales_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{LAB_SCHEMA}.sales_delta")

print("Written: JSON, Parquet, and Delta (managed table sales_delta)")

Written: JSON, Parquet, and Delta (managed table sales_delta)


In [0]:
# ─── Measure on-disk size for each format ─────────────────────────────

def folder_size_bytes(path):
    total = 0
    for f in dbutils.fs.ls(path):
        if f.isDir():
            total += folder_size_bytes(f.path)
        else:
            total += f.size
    return total

# CSV size
csv_size = dbutils.fs.ls(csv_path)[0].size

# JSON size
json_size = folder_size_bytes(json_path)

# Parquet size
parquet_size = folder_size_bytes(parquet_path)

# Delta size (from Delta metadata instead of traversing storage path)
delta_detail = spark.sql(
    f"DESCRIBE DETAIL {CATALOG}.{LAB_SCHEMA}.sales_delta"
)

delta_size = delta_detail.collect()[0]["sizeInBytes"]

print(f"{'Format':<10} {'Size (MB)':>12} {'Compression vs CSV':>22}")

for name, size in [
    ("CSV", csv_size),
    ("JSON", json_size),
    ("Parquet", parquet_size),
    ("Delta", delta_size)
]:
    ratio = csv_size / size if size else 0
    print(f"{name:<10} {size/1e6:>12.1f} {ratio:>21.2f}x")

Format        Size (MB)     Compression vs CSV
CSV               976.4                  1.00x
JSON             2530.9                  0.39x
Parquet           166.0                  5.88x
Delta             118.5                  8.24x


In [0]:
# ─── Measure read + aggregation time for each format ───────────────────────────
# Same query against each format: total Sales by Category. This forces a full
# scan of the data, so the timing difference reflects the storage format
# itself, not query complexity.

def timed_aggregation(read_fn, label):
    start = time.time()
    df = read_fn()
    result = df.groupBy("Category").sum("Sales").collect()
    elapsed = time.time() - start
    print(f"{label:<10} {elapsed:>8.2f}s")
    return elapsed

print(f"{'Format':<10} {'Time':>8}")
csv_time     = timed_aggregation(lambda: spark.read.option("header","true").option("inferSchema","true").csv(csv_path), "CSV")
json_time    = timed_aggregation(lambda: spark.read.json(json_path), "JSON")
parquet_time = timed_aggregation(lambda: spark.read.parquet(parquet_path), "Parquet")
delta_time   = timed_aggregation(lambda: spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_delta"), "Delta")

Format         Time
CSV           20.41s
JSON          31.17s
Parquet        2.04s
Delta          3.03s


### Observe

- **Size**: Parquet and Delta should both be dramatically smaller than the raw CSV (often 3–10x smaller, depending on how repetitive the column values are) — JSON is usually *larger* than CSV, not smaller, because repeating every field name on every single record adds real overhead with zero compression to offset it.
- **Speed**: Parquet and Delta should read+aggregate noticeably faster than CSV or JSON, even though this query (`groupBy("Category").sum("Sales")`) only touches 2 of the 25 columns.

### Why did this happen?

This is **column pruning** in action: Parquet's columnar layout means Spark can read *only* the `Category` and `Sales` columns off disk and skip the other 23 entirely. CSV and JSON are row-based — Spark has to read (and for CSV, re-parse as text) every single column of every single row even though 23 of them are irrelevant to this query. Delta gets the same columnar benefit as Parquet (it *is* Parquet underneath) plus stores extra statistics in its transaction log that let it skip whole files that can't contain relevant data at all.

### Azure Databricks verification

- **Catalog Explorer → your schema → Tables → `sales_delta`** — open the table details. You'll see it's a **Delta** table with a real storage `Location`. Click through to that location in the Volume/file browser and you'll see actual `.parquet` data files sitting alongside a `_delta_log/` folder — that log is what makes it a Delta table and not just "a folder of Parquet files."
- **Volume browser** → open the `formats/sales_parquet` and `formats/sales_json` folders directly and compare file sizes visually against what you measured above.

### Spark UI verification

Open the **Spark UI** (cluster → Spark UI tab, or the notebook's own "View" link on a completed cell) → **Jobs** or **SQL/DataFrame** tab → find the 4 aggregation jobs you just ran. For each, check:
- **Input Size** — should be far smaller for Parquet/Delta than CSV/JSON (this is column pruning made visible as an actual metric, not just faster wall-clock time)
- **Number of Tasks** — CSV/JSON often need more tasks to parse the same volume of row-based text

### Production use case

This exact comparison is why almost every real pipeline looks like: `raw file (CSV/JSON) → Bronze (Delta, as-is)`, then `Bronze → Silver (Delta, cleaned)`, then `Silver → Gold (Delta, aggregated)`. Every hop after the very first lands as Delta specifically because of what you just measured — every downstream read is faster and smaller.

---
## Section 5 — Partitioning Strategy

### Concept

**Partitioning** means physically splitting a table's files into separate folders based on the value of one or more columns — e.g. all rows where `Year=2024` and `Month=6` live together in a folder literally named `Year=2024/Month=6/`. When a query filters on a partition column, Spark can look at the folder names alone and **skip entire folders it knows can't match** — it never even opens those files. That's **partition pruning**, and it's a completely different, much coarser mechanism than the column pruning you just saw in Section 4 (column pruning skips *columns within* a file; partition pruning skips *whole files* based on folder names).

### Code — write partitioned by Year and Month

In [0]:
# partitionBy() controls the folder structure on disk — every distinct
# (Year, Month) combination gets its own subfolder.
(sales_df.write
    .mode("overwrite")
    .partitionBy("Year", "Month")
    .saveAsTable(f"{CATALOG}.{LAB_SCHEMA}.sales_partitioned"))

print("Written: sales_partitioned, partitioned by Year, Month")

Written: sales_partitioned, partitioned by Year, Month


In [0]:
# ─── Show partition structure using metadata (works with Unity Catalog) ───

print("Partition Columns:")
spark.sql(f"""
DESCRIBE DETAIL {CATALOG}.{LAB_SCHEMA}.sales_partitioned
""").select("partitionColumns").show(truncate=False)

print("\nSample Partitions:")
spark.sql(f"""
SHOW PARTITIONS {CATALOG}.{LAB_SCHEMA}.sales_partitioned
""").show(50, truncate=False)

Partition Columns:
+----------------+
|partitionColumns|
+----------------+
|[Year, Month]   |
+----------------+


Sample Partitions:
+----+-----+
|Year|Month|
+----+-----+
|2022|10   |
|2024|7    |
|2024|12   |
|2023|8    |
|2023|9    |
|2025|5    |
|2024|3    |
|2025|6    |
|2023|7    |
|2026|5    |
|2025|10   |
|2024|5    |
|2026|12   |
|2022|2    |
|2022|7    |
|2023|6    |
|2026|2    |
|2025|9    |
|2025|7    |
|2024|9    |
|2024|10   |
|2026|9    |
|2022|11   |
|2026|10   |
|2022|3    |
|2024|2    |
|2026|7    |
|2023|3    |
|2025|8    |
|2026|3    |
|2023|2    |
|2023|11   |
|2025|3    |
|2023|4    |
|2023|5    |
|2026|6    |
|2025|12   |
|2024|1    |
|2026|11   |
|2024|11   |
|2025|4    |
|2022|1    |
|2023|10   |
|2026|8    |
|2024|6    |
|2025|11   |
|2022|5    |
|2025|1    |
|2022|6    |
|2026|1    |
+----+-----+
only showing top 50 rows


In [0]:
# ─── Query WITH a partition filter vs WITHOUT one ──────────────────────────────
partitioned_df = spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_partitioned")

start = time.time()
filtered_count = partitioned_df.filter("Year = 2024 AND Month = 6").count()
filtered_time = time.time() - start

start = time.time()
unfiltered_count = partitioned_df.count()
unfiltered_time = time.time() - start

print(f"WITH partition filter (Year=2024, Month=6): {filtered_count:,} rows in {filtered_time:.2f}s")
print(f"WITHOUT filter (full table)               : {unfiltered_count:,} rows in {unfiltered_time:.2f}s")

WITH partition filter (Year=2024, Month=6): 82,151 rows in 0.49s
WITHOUT filter (full table)               : 5,000,000 rows in 0.27s


In [0]:
# ─── Prove pruning happened, don't just infer it from timing ──────────────────
partitioned_df.filter("Year = 2024 AND Month = 6").explain(True)
# Look for "PartitionFilters" in the physical plan output below — that line
# lists exactly which partition predicates Spark used to skip folders before
# reading anything. Compare against the plan for the unfiltered count()
# above, which has no PartitionFilters at all (it must scan every folder).

== Parsed Logical Plan ==
'Filter (('Year = 2024) AND ('Month = 6))
+- SubqueryAlias naya_catalog.day4_lab.sales_partitioned
   +- Relation naya_catalog.day4_lab.sales_partitioned[OrderID#4382,CustomerID#4383,OrderDate#4384,ShipDate#4385,Year#4386,Month#4387,Day#4388,Country#4389,State#4390,City#4391,Region#4392,Category#4393,SubCategory#4394,ProductName#4395,Brand#4396,Quantity#4397,UnitPrice#4398,Discount#4399,Sales#4400,Profit#4401,PaymentMethod#4402,OrderPriority#4403,ShippingMode#4404,CustomerSegment#4405,Returned#4406] parquet

== Analyzed Logical Plan ==
OrderID: string, CustomerID: string, OrderDate: date, ShipDate: date, Year: int, Month: int, Day: int, Country: string, State: string, City: string, Region: string, Category: string, SubCategory: string, ProductName: string, Brand: string, Quantity: int, UnitPrice: double, Discount: double, Sales: double, Profit: double, PaymentMethod: string, OrderPriority: string, ShippingMode: string, CustomerSegment: string, Returned: string

### Observe

- The folder listing shows real `Year=2022/` ... `Year=2026/` folders, each containing `Month=1/` through `Month=12/` subfolders — Delta wrote the partition structure exactly as named in `partitionBy("Year", "Month")`.
- The filtered query (one Year + one Month) should run **much faster** than counting the whole table, even though the whole table isn't *that* much bigger in row count terms.
- In the `.explain(True)` output, look for a line containing **`PartitionFilters: [isnotnull(Year#...), (Year#... = 2024), ...]`** — that's Spark telling you explicitly which folders it skipped before reading anything.

### Why did this happen?

Partition pruning happens at the **file-listing stage**, before Spark even opens a single Parquet file. With `Year=2024 AND Month=6` in the filter, Spark computes "the only folder that can possibly satisfy this is `Year=2024/Month=6/`" and lists *only* that folder's files — every other `Year=`/`Month=` folder is never even touched. The unfiltered count has to list and read every partition folder that exists.

### Azure Databricks verification

**Catalog Explorer → your schema → Tables → `sales_partitioned` → open the storage location** (or browse the Volume path directly) — you'll see the real `Year=YYYY/Month=M/` folder hierarchy on disk, plus a `_delta_log/` folder at the table root recording every write as a JSON commit. This is the physical proof of what `partitionBy()` did.

### Spark UI verification

Open the **Spark UI → SQL/DataFrame tab** and find the two `count()` jobs from above. Compare:
- **Number of Tasks** — the filtered job should have far fewer tasks (roughly proportional to how many partition folders it actually touched vs. all of them)
- **Input Size** — should be dramatically smaller for the filtered job, since most of the table's files were never read

### Production use case

Gold-layer fact tables in a real warehouse are almost always partitioned by date (`Year`/`Month`, or sometimes just a single `date` column) precisely because BI dashboards overwhelmingly filter by a recent date range ("last 30 days", "this quarter") — partition pruning means those dashboard queries only ever touch a small slice of the table's total history, no matter how many years of data the table holds.

---
## Section 6 — Performance Levers: File Sizing

### Concept

**Too many small files** is one of the most common real-world Spark performance killers. Every file Spark opens has fixed overhead (listing it, opening a connection to it, scheduling a task for it) — if your data is spread across thousands of tiny files instead of a sensible number of larger ones, that per-file overhead can dominate the actual work being done. **Too few, overly large files** has the opposite problem: less parallelism (fewer tasks than available cores means some cores sit idle), and a single slow/failed task takes longer to retry. The sweet spot Databricks generally recommends is roughly **128MB–1GB per file** — small enough for good parallelism, large enough that per-file overhead is negligible.

### Code — create the small-files problem on purpose

In [0]:

# Create many small files
many_small_path = f"{CATALOG}.{LAB_SCHEMA}.sales_many_small_files"

sales_df.repartition(200) \
        .write \
        .mode("overwrite") \
        .saveAsTable(many_small_path)

# Get file count from Delta metadata
detail = spark.sql(f"DESCRIBE DETAIL {many_small_path}")

file_count = detail.collect()[0]["numFiles"]

print(f"Files written: {file_count}")


Files written: 200


In [0]:
# ─── Measure a query against the many-small-files table ───────────────────────
start = time.time()
spark.table(many_small_path).groupBy("Category").sum("Sales").collect()
many_small_time = time.time() - start
print(f"Query time against {file_count} small files: {many_small_time:.2f}s")

Query time against 200 small files: 6.43s


In [0]:

# ─── Now compact into fewer, larger files ──────────────────────────────────────
try:
    spark.sql(f"OPTIMIZE {many_small_path}")
    print("OPTIMIZE completed.")
except Exception as e:
    print(f"OPTIMIZE not available here ({e}); falling back to a manual repartition+rewrite.")
    sales_df.repartition(8).write.mode("overwrite").saveAsTable(many_small_path)

# Get file count from Delta metadata instead of listing the storage path
detail = spark.sql(f"DESCRIBE DETAIL {many_small_path}")

file_count_after = detail.collect()[0]["numFiles"]

print(f"Files after compaction: {file_count_after}")


OPTIMIZE completed.
Files after compaction: 3


In [0]:
start = time.time()
spark.table(many_small_path).groupBy("Category").sum("Sales").collect()
compacted_time = time.time() - start
print(f"Query time after compaction ({file_count_after} files): {compacted_time:.2f}s")
print(f"\nBefore: {file_count} files, {many_small_time:.2f}s")
print(f"After : {file_count_after} files, {compacted_time:.2f}s")

Query time after compaction (3 files): 1.34s

Before: 200 files, 6.43s
After : 3 files, 1.34s


### Observe

- The many-small-files table should show a noticeably higher file count than makes sense for its data volume.
- The same aggregation query should run **slower** against the many-small-files version than after compaction, even though it's reading the exact same underlying rows.

### Why did this happen?

Each of those 200 small files needs its own task: listing it, opening it, scheduling and starting a task for it, closing it out. When a file is small, that fixed overhead is a *large fraction* of the total time spent on that file — you're paying "per-file tax" 200 times over instead of a handful of times. After `OPTIMIZE`, the same data lives in far fewer, appropriately-sized files, so the per-file overhead shrinks dramatically relative to actual data processing time.

### Azure Databricks verification

**Catalog Explorer → `sales_many_small_files` → table details → storage location** — browse the folder before and after running the `OPTIMIZE` cell and count the `.parquet` files directly; you should see a visible drop.

### Spark UI verification

Compare the two `groupBy(...).sum(...)` jobs in **Spark UI → Jobs**: the many-small-files job should show a much higher **Number of Tasks** for roughly the same total **Input Size** — that mismatch (many tasks, same data volume) is the small-files problem made visible as a metric.

### Production use case

Streaming and micro-batch pipelines (like Autoloader ingesting file-by-file) are notorious for accumulating small files over time, since each micro-batch tends to write its own small file. Production Delta pipelines schedule **`OPTIMIZE`** to run periodically (nightly, or after N writes) specifically to compact these away before they degrade downstream query performance.

---
## Section 7 — Caching

### Concept

Spark is **lazy** — a DataFrame is just a *plan* until an action (`count()`, `collect()`, `.show()`, etc.) forces it to actually run. Normally, every action re-executes the plan from scratch, including re-reading the source data. **`.cache()`** tells Spark: the *first* time this DataFrame is computed, keep the result in memory (or spill to disk if it doesn't fit) so any *later* action against the same DataFrame reuses that stored result instead of recomputing everything.

### Code

In [0]:
delta_df = spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_delta")

# .cache() only marks the DataFrame for caching — it does NOT cache anything
# yet. Nothing is materialized until the first action runs.
delta_df.cache()

# First run: this is the "cold" run. It reads from disk AND populates the cache.
start = time.time()
first_result = delta_df.groupBy("Category").sum("Sales").collect()
first_run_time = time.time() - start
print(f"First run (cold, populates cache): {first_run_time:.2f}s")

First run (cold, populates cache): 24.07s


In [0]:
# Second run: same query, same DataFrame -- should hit the in-memory cache
# instead of re-reading from disk.
start = time.time()
second_result = delta_df.groupBy("Category").sum("Sales").collect()
second_run_time = time.time() - start
print(f"Second run (from cache): {second_run_time:.2f}s")
print(f"\nSpeedup: {first_run_time / max(second_run_time, 0.001):.1f}x faster")

delta_df.unpersist()  # good hygiene -- free the cached memory once you're done with it

Second run (from cache): 0.57s

Speedup: 42.0x faster


DataFrame[OrderID: string, CustomerID: string, OrderDate: date, ShipDate: date, Year: int, Month: int, Day: int, Country: string, State: string, City: string, Region: string, Category: string, SubCategory: string, ProductName: string, Brand: string, Quantity: int, UnitPrice: double, Discount: double, Sales: double, Profit: double, PaymentMethod: string, OrderPriority: string, ShippingMode: string, CustomerSegment: string, Returned: string]

### Observe

The second run should be noticeably faster than the first — often dramatically so, since it skips reading from storage entirely and works directly from the copy already sitting in cluster memory.

### Why did this happen?

The first `collect()` is the one that actually triggers the read from Delta storage — and because `.cache()` was called beforehand, Spark also stores the resulting data in memory as a side effect of that first execution. The second `collect()` reuses that in-memory copy instead of touching storage again.

### Azure Databricks verification

Not much to see in Catalog Explorer for this one — caching is a cluster-memory concept, not a storage-layer one.

### Spark UI verification

Open **Spark UI → Storage tab** right after the first run — you should see `sales_delta`'s cached DataFrame listed there with its size in memory and the fraction of partitions cached. Then compare the two `groupBy` jobs under **Jobs**: the second job's stages should show tasks reading from cached memory rather than from the underlying storage.

### Production use case

Caching is most valuable when the **same DataFrame is reused multiple times** in one session — e.g. a cleaned dimension table (like a customer or product lookup) that gets joined against several different fact tables in the same notebook or job. Caching it once avoids re-reading and re-cleaning it on every join. It's *not* useful for a DataFrame you only touch once — in that case caching just adds overhead for no benefit, and on a memory-constrained cluster it can even evict something more useful.

---
## Section 8 — Clustering

### Concept

**Partitioning vs. clustering — the difference that trips people up most:**

- **Partitioning** (Section 5) physically splits data into separate *folders* based on column values. It's coarse-grained by design — you should only partition by columns with a *small* number of distinct values (dozens, not thousands), because every distinct value becomes its own folder. Partition too finely and you recreate the small-files problem from Section 6.
- **Clustering** (via `ZORDER`, or **Liquid Clustering** on newer Databricks runtimes) doesn't create folders at all — it **co-locates related data within the existing files**, physically ordering/grouping rows so that rows with similar values in the clustered column end up sitting near each other on disk. This lets Delta's data-skipping statistics skip whole *files* for a filter on that column, without exploding your folder count the way partitioning by a high-cardinality column would.

In short: **partition by a handful of coarse values you always filter by (like Year/Month); cluster/Z-ORDER by higher-cardinality columns you frequently filter by but that don't make sense as partition folders (like `Category` or `CustomerID`).**

### Code

In [0]:
# Baseline: filter on Category BEFORE clustering
start = time.time()
before_count = spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_delta").filter("Category = 'Electronics'").count()
before_time = time.time() - start
print(f"Before ZORDER — filter on Category: {before_count:,} rows in {before_time:.2f}s")

Before ZORDER — filter on Category: 625,230 rows in 1.09s


In [0]:
# ZORDER BY co-locates rows with similar Category values within files.
# Availability depends on your DBR/Unity Catalog setup -- if your workspace
# uses newer Liquid Clustering instead, the equivalent is:
#   ALTER TABLE ... CLUSTER BY (Category)
# Try ZORDER first; fall back gracefully if it isn't supported.
try:
    spark.sql(f"OPTIMIZE {CATALOG}.{LAB_SCHEMA}.sales_delta ZORDER BY (Category)")
    print("ZORDER BY (Category) completed.")
except Exception as e:
    print(f"ZORDER not available here ({e}) -- check whether Liquid Clustering "
          f"(ALTER TABLE ... CLUSTER BY) is supported on your DBR/UC setup instead.")

ZORDER BY (Category) completed.


In [0]:
start = time.time()
after_count = spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_delta").filter("Category = 'Electronics'").count()
after_time = time.time() - start
print(f"After ZORDER — filter on Category: {after_count:,} rows in {after_time:.2f}s")
print(f"\nBefore: {before_time:.2f}s   After: {after_time:.2f}s")

After ZORDER — filter on Category: 625,230 rows in 1.15s

Before: 1.09s   After: 1.15s


### Observe

The `Category = 'Electronics'` filter should run faster after `ZORDER BY (Category)` than before — how much faster depends on how spread out `Electronics` rows were across files beforehand.

### Why did this happen?

Before clustering, rows for `Electronics` were scattered essentially randomly across every data file (since the table was originally written without any ordering by `Category`), so Spark had to open nearly every file to find them. After `ZORDER BY (Category)`, rows with the same `Category` value are physically grouped together within a much smaller number of files — Delta's file-level min/max statistics can then say "this file only contains Furniture and Books, skip it entirely" for far more files than before.

### Azure Databricks verification

**Catalog Explorer → `sales_delta` → table details → History** (or `DESCRIBE HISTORY`) — you should see an `OPTIMIZE` operation with `ZORDER BY` recorded as a table version, confirming it actually ran and giving you the file counts it touched.

### Spark UI verification

Compare the before/after filter jobs in **Spark UI → Jobs → Stage Details**: look at how many files/tasks were actually scanned. After clustering, you should see evidence of **data skipping** — fewer files read for the same filter, which is the file-skipping mechanism clustering enables, distinct from the folder-level skipping partitioning gives you.

### Production use case

Gold tables are often partitioned by date (coarse, always-filtered) **and** Z-ORDERed/clustered by a second, higher-cardinality column that's frequently filtered but would make a terrible partition column — e.g. a fact table partitioned by `Year`/`Month` and Z-ORDERed by `CustomerID` or `ProductID`, so both "last quarter's data" and "this one customer's orders" queries are fast, without either column exploding the folder count.

---
## Section 9 — Delta Lake Features

### Concept

Everything so far has used the *storage* half of Delta (Parquet + partitioning). This section covers the *transaction log* half — the features that only exist because Delta records every change as a versioned, atomic commit: `UPDATE`, `DELETE`, `MERGE` (upsert), schema evolution, and time travel. None of these are possible on plain Parquet.

### UPDATE — change existing rows in place

In [0]:
# Mark all 'High' priority orders that were paid via Cash on Delivery as
# 'Critical' priority -- a realistic "fix a business rule after the fact" update.
spark.sql(f"""
    UPDATE {CATALOG}.{LAB_SCHEMA}.sales_delta
    SET OrderPriority = 'Critical'
    WHERE OrderPriority = 'High' AND PaymentMethod = 'Cash on Delivery'
""")
print("UPDATE complete.")

UPDATE complete.


### DELETE — remove rows

In [0]:
# Remove test/garbage rows as an example -- here, orders with zero quantity
# (shouldn't exist in real data, but demonstrates the mechanism).
before_delete = spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_delta").count()

spark.sql(f"""
    DELETE FROM {CATALOG}.{LAB_SCHEMA}.sales_delta
    WHERE Quantity <= 0
""")

after_delete = spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_delta").count()
print(f"Rows before: {before_delete:,}   Rows after: {after_delete:,}   Deleted: {before_delete - after_delete:,}")

Rows before: 5,000,000   Rows after: 5,000,000   Deleted: 0


### MERGE — upsert (insert new rows, update matching existing ones) in one atomic operation

In [0]:
# Simulate a small incoming batch: 1 brand-new order, 1 update to an existing order.
from pyspark.sql import Row

sample_existing_id = spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_delta").select("OrderID").first()["OrderID"]

incoming_batch = spark.createDataFrame([
    Row(OrderID="ORD-99999999", CustomerID="CUST-000001", Returned="No"),   # new row
    Row(OrderID=sample_existing_id, CustomerID="CUST-000001", Returned="Yes"),  # updates an existing row
])

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{LAB_SCHEMA}.incoming_batch")
incoming_batch.write.saveAsTable(f"{CATALOG}.{LAB_SCHEMA}.incoming_batch")

spark.sql(f"""
    MERGE INTO {CATALOG}.{LAB_SCHEMA}.sales_delta AS target
    USING {CATALOG}.{LAB_SCHEMA}.incoming_batch AS source
    ON target.OrderID = source.OrderID
    WHEN MATCHED THEN UPDATE SET target.Returned = source.Returned
    WHEN NOT MATCHED THEN INSERT (OrderID, CustomerID, Returned) VALUES (source.OrderID, source.CustomerID, source.Returned)
""")
print("MERGE complete -- 1 row updated, 1 row inserted.")

MERGE complete -- 1 row updated, 1 row inserted.


### Schema Evolution — add a new column without rewriting the whole table

In [0]:
from pyspark.sql.functions import lit

# Add a new column that didn't exist in the original write.
# mergeSchema=true tells Delta "this is an intentional schema change, accept it."
enriched_df = sales_df.withColumn("LoyaltyTier", lit("Standard"))

enriched_df.write \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{CATALOG}.{LAB_SCHEMA}.sales_delta")

spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_delta").printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- ShipDate: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- OrderPriority: string (nullable = true)
 |-- ShippingMode: string (nullable = true)
 |-- CustomerSegment: string (nullable = true)
 |-- Returned: string (nullable = t

### Time Travel — query (or restore) a previous version of the table

In [0]:
# DESCRIBE HISTORY shows every commit ever made to this table -- every write,
# UPDATE, DELETE, MERGE, and schema change from this section is recorded as
# its own numbered version.
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{LAB_SCHEMA}.sales_delta"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-07-10T10:26:22Z,142696759188881,raj.sharma01@npmentorskool.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(81287692045414),0706-135832-22z9pk7m,4,WriteSerializable,true,"Map(numFiles -> 8, numOutputRows -> 5000000, numOutputBytes -> 118572078)",null,Databricks-Runtime/16.4.x-scala2.12
4,2026-07-10T10:25:23Z,142696759188881,raj.sharma01@npmentorskool.onmicrosoft.com,MERGE,"Map(predicate -> [""(OrderID#15443 = OrderID#15471)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(81287692045414),0706-135832-22z9pk7m,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 7513, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4972, materializeSourceTimeMs -> 3, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2684, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2229)",null,Databricks-Runtime/16.4.x-scala2.12
3,2026-07-10T10:25:07Z,142696759188881,raj.sharma01@npmentorskool.onmicrosoft.com,DELETE,"Map(predicate -> [""(Quantity#14691 <= 0)""])",null,List(81287692045414),0706-135832-22z9pk7m,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 283, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 280, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
2,2026-07-10T10:24:59Z,142696759188881,raj.sharma01@npmentorskool.onmicrosoft.com,UPDATE,"Map(predicate -> [""((OrderPriority#12815 = High) AND (PaymentMethod#12814 = Cash on Delivery))""])",null,List(81287692045414),0706-135832-22z9pk7m,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 9933, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2402, numAddedFiles -> 1, numUpdatedRows -> 208917, numAddedBytes -> 5564663, rewriteTimeMs -> 7512)",null,Databricks-Runtime/16.4.x-scala2.12
1,2026-07-10T10:21:06Z,142696759188881,raj.sharma01@npmentorskool.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [""Category""], batchId -> 0)",null,List(81287692045414),0706-135832-22z9pk7m,0,SnapshotIsolation,false,"Map(numRemovedFiles -> 8, numRemovedBytes -> 118549976, p25FileSize -> 124185815, numDeletionVectorsRemoved -> 0, minFileSize -> 124185815, numAddedFiles -> 1, maxFileSize -> 124185815, p75FileSize -> 124185815, p50FileSize -> 124185815, numAddedBytes -> 124185815)",null,Databricks-Runtime/16.4.x-scala2.12
0,2026-07-10T09:56:28Z,142696759188881,raj.sharma01@npmentorskool.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(81287692045414),0706-135832-22z9pk7m,null,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> 5000000, numOutputBytes -> 118549976)",null,Databricks-Runtime/16.4.x-scala2.12


In [0]:
# Query the table AS IT EXISTED at version 0 -- before any of this section's
# UPDATE/DELETE/MERGE/schema-evolution changes were applied.
original_version = spark.sql(f"SELECT COUNT(*) AS n FROM {CATALOG}.{LAB_SCHEMA}.sales_delta VERSION AS OF 0").collect()[0]["n"]
current_version   = spark.table(f"{CATALOG}.{LAB_SCHEMA}.sales_delta").count()

print(f"Version 0 (original write) row count : {original_version:,}")
print(f"Current version row count             : {current_version:,}")

Version 0 (original write) row count : 5,000,000
Current version row count             : 10,000,001


### Observe

- `DESCRIBE HISTORY` lists a growing sequence of versions — one per write/UPDATE/DELETE/MERGE/schema-change you ran in this section, each with its own version number, timestamp, and operation type.
- The row count at `VERSION AS OF 0` differs from the current row count, because the DELETE and MERGE and appended schema-evolution batch changed the row count after version 0 was written.

### Why did this happen?

Every Delta write — no matter which SQL statement caused it — is recorded as a new, immutable, numbered entry in the `_delta_log/`. `VERSION AS OF` simply tells Delta "reconstruct the table exactly as it looked at that log entry," by replaying only the log entries up to that point. Nothing is ever destructively overwritten; old Parquet files referenced by earlier versions stick around (until a retention-based cleanup like `VACUUM` removes ones no longer referenced by any recent version).

### Azure Databricks verification

**Catalog Explorer → `sales_delta` → History tab** shows the exact same version list as `DESCRIBE HISTORY`, in the UI — a fast way to audit what changed and when without writing SQL.

### Production use case

- **MERGE** is the standard pattern for CDC-style incremental loads into Silver — new/changed source rows get upserted without duplicating unchanged ones (this is the same pattern Day 2's Lakeflow Connect pipeline uses under the hood).
- **Schema evolution** lets a source system add a field (like a new `LoyaltyTier` attribute) without your whole pipeline breaking or needing a manual migration.
- **Time travel** is what makes "a bad job run corrupted last night's Gold table" a `RESTORE TABLE ... TO VERSION AS OF <n>` away from fixed, instead of a restore-from-backup emergency.

---
## Section 10 — Best Practices

### When to use each storage format

| Format | Use it for |
|---|---|
| **CSV** | Landing raw data exactly as a source system delivered it (Bronze intake only). Never as a working/query format. |
| **JSON** | Semi-structured API responses, nested/variable schemas. Still a landing format, not a working one. |
| **Parquet** | Intermediate analytical datasets where you don't need Delta's transaction log (rare in this course — usually you'd just use Delta). |
| **Delta** | Everything from Bronze onward in a real pipeline — Bronze, Silver, and Gold. The default choice once data is inside the lakehouse. |

### Partitioning

- **Partition when**: the column has low-to-moderate cardinality (roughly tens to low hundreds of distinct values) and your queries commonly filter on it — date columns (`Year`/`Month`) are the classic case.
- **Don't partition when**: the column is high-cardinality (`CustomerID`, `OrderID`) — that creates the small-files problem from Section 6, one tiny folder per distinct value. Use clustering/Z-ORDER for those instead.

### Ideal file sizes

Roughly **128MB–1GB per file** — small enough for good task parallelism, large enough that per-file overhead (listing, opening, scheduling) stays negligible relative to actual work done.

### Caching recommendations

Cache a DataFrame only when it's **reused multiple times** in the same session/job (e.g. a shared dimension joined against several fact tables). Always `.unpersist()` when done. Don't cache something you touch exactly once — it adds overhead for zero benefit and can evict more useful cached data on a memory-constrained cluster.

### Clustering recommendations

Use `ZORDER`/Liquid Clustering on higher-cardinality columns you frequently filter by but that don't make sense as partition folders. It's a complement to partitioning, not a replacement — real Gold tables commonly use both together (partition by date, cluster by a business key).

---
## Section 11 — Final Summary

| Lever | What it controls | Key mechanism | What you measured in this lab |
|---|---|---|---|
| **Storage Format** | How data is physically encoded on disk | Row-based (CSV/JSON) vs columnar (Parquet/Delta); Delta adds a transaction log on top of Parquet | Section 4: Parquet/Delta smaller AND faster than CSV/JSON for the same query |
| **Partitioning** | Which *folders* a query has to open | Partition pruning skips whole folders based on filter columns matching folder names | Section 5: filtered query far faster than full scan, proven via `.explain()`'s `PartitionFilters` |
| **File Sizing** | How many files a query has to open | Per-file overhead (listing/opening/task-scheduling) dominates when files are too small | Section 6: same data, same query, slower with 200 small files than after `OPTIMIZE` |
| **Caching** | Whether repeated queries re-read from storage | First action materializes + stores in cluster memory; later actions reuse it | Section 7: second run of the identical query dramatically faster than the first |
| **Clustering** | Which *files* (not folders) a query has to open | Co-locates similar values within files so file-level stats can skip files, without exploding folder count | Section 8: `Category` filter faster after `ZORDER BY (Category)` |

```
Storage Format
      ↓
Partitioning
      ↓
Performance (file sizing + caching + clustering)
      ↓
Fast Spark Queries
```

Each lever compounds with the others: the right storage format makes partition pruning and clustering possible in the first place; the right partitioning strategy keeps file sizes sane; caching layers on top of whatever the storage/partitioning/clustering choices already made fast.

---
## Section 12 — Final Hands-On Challenge

No code is given for this one — apply what you just learned yourself.

**Your task:**

1. Pick a **different partition column combination** than `Year`/`Month` (for example: `Country` alone, or `Country` + `Category`) and write a new partitioned Delta table from `sales_df`.
2. Pick a realistic query a business user might actually run against your new partitioning choice, and measure its execution time **with** a partition filter and **without** one — same pattern as Section 5.
3. Enable caching on a DataFrame you query more than once, and measure the before/after timing — same pattern as Section 7.
4. Compare your results against this notebook's Section 5/7 numbers. Are they better, worse, or about the same? Would you recommend your new partition column combination for a real Gold table querying this data — why or why not?

Fill in your findings below.

---

### Your Observations (fill this in)

**Partition columns I chose:** _______________________

**Why I chose them (which queries would benefit):** _______________________

**Execution time WITH partition filter:** _______________________

**Execution time WITHOUT partition filter:** _______________________

**Did caching help my second query? By how much?** _______________________

**Would I recommend this partitioning strategy for production? Why or why not?** _______________________

---

## Cleanup (optional)

This lab created several scratch tables under `{CATALOG}.{LAB_SCHEMA}`. If you don't need them anymore:
```sql
DROP SCHEMA IF EXISTS YOUR_CATALOG.day4_lab CASCADE;
```